In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import itertools
import seaborn as sns
import time
import pandas as pd
import cv2
import torchvision.transforms.functional as TF

In [ ]:
ROOT_DIR = "C:\Users\omarh\OneDrive - Georgia Institute of Technology\openEDS2019"

**Dataloader**

In [ ]:
class EyeBoundingBoxDataset(Dataset):
    def __init__(self, subject_ids, root_dir, transform=None, apply_preprocessing=True):
        self.root_dir = root_dir
        self.subject_ids = subject_ids
        self.transform = transform
        self.apply_preprocessing = apply_preprocessing  # Whether to apply Gamma + Stretch
        self.data = []

        # Precompute gamma LUT once to save time
        gamma = 0.8
        self.gamma_LUT = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)

        for subject_id in self.subject_ids:
            subject_dir = os.path.join(root_dir, 'openEDS', 'openEDS', subject_id)
            bbox_file = os.path.join(root_dir, 'bbox', 'bbox', f"{subject_id}.txt")
            with open(bbox_file, 'r') as f:
                bboxes = [list(map(float, line.strip().split())) for line in f.readlines()]
            for i in range(len(bboxes) - 1):
                img0_path = os.path.join(subject_dir, f"{i}.png")
                img1_path = os.path.join(subject_dir, f"{i+1}.png")
                self.data.append((img0_path, img1_path, bboxes[i+1]))

    def __len__(self):
        return len(self.data)

    def preprocess_image(self, img_path):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Could not load image from: {img_path}")

        # 1) Apply Gamma Correction
        img = cv2.LUT(img, self.gamma_LUT)

        # 2) Apply Percentile Stretch via Vectorized LUT
        p_low, p_high = np.percentile(img, (1, 99))
        lut_indices = np.arange(256)
        stretch_LUT = np.clip((lut_indices - p_low) * (255.0 / max(p_high - p_low, 1)), 0, 255).astype(np.uint8)
        img = cv2.LUT(img, stretch_LUT)

        return img

    def load_image(self, img_path):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Could not load image from: {img_path}")
        return img

    def __getitem__(self, idx):
        img0_path, img1_path, bbox = self.data[idx]

        if self.apply_preprocessing:
            img0 = self.preprocess_image(img0_path)
            img1 = self.preprocess_image(img1_path)
        else:
            img0 = self.load_image(img0_path)
            img1 = self.load_image(img1_path)

        # Convert numpy arrays (H, W) -> (1, H, W) torch.Tensor
        img0 = torch.from_numpy(img0).unsqueeze(0).float() / 255.0  # scale 0-1
        img1 = torch.from_numpy(img1).unsqueeze(0).float() / 255.0

        if self.transform:
            img0 = self.transform(img0)
            img1 = self.transform(img1)

        diff = img1 - img0
        input_tensor = torch.cat((img1, diff), dim=0)  # shape (2, H, W)

        target = torch.tensor(bbox, dtype=torch.float32)
        return input_tensor, target

**Model Definition**

In [ ]:
class LightweightBBoxCNN(nn.Module):
    def __init__(self, hidden_size=64):  # now configurable
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, hidden_size),  # updated hidden size
            nn.ReLU(),
            nn.Linear(hidden_size, 4)  # 4 = xmin, xmax, ymin, ymax
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x

**Split by Subject**

In [ ]:
def split_subjects_and_save(root_dir, output_file='subject_split.txt', seed=42):
    random.seed(seed)
    subject_path = os.path.join(root_dir, 'openEDS', 'openEDS')
    all_subjects = sorted([d for d in os.listdir(subject_path) if d.startswith('S_') and os.path.isdir(os.path.join(subject_path, d))])

    random.shuffle(all_subjects)
    n_total = len(all_subjects)
    n_train = int(0.7 * n_total)
    n_val = int(0.2 * n_total)

    train_subjects = all_subjects[:n_train]
    val_subjects = all_subjects[n_train:n_train + n_val]
    test_subjects = all_subjects[n_train + n_val:]

    with open(output_file, 'w') as f:
        f.write("Training Subjects:\n")
        for s in train_subjects:
            f.write(f"{s}\n")
        f.write("\nValidation Subjects:\n")
        for s in val_subjects:
            f.write(f"{s}\n")
        f.write("\nTest Subjects:\n")
        for s in test_subjects:
            f.write(f"{s}\n")

    print(f"Subject split saved to {output_file}")
    return train_subjects, val_subjects, test_subjects

**Random Augmentations**

In [ ]:
class RandomAugmentations:
    def __init__(self, p=0.2):
        self.p = p

    def __call__(self, x):
        # Always tensor input [C, H, W]
        if random.random() < self.p:
            # Horizontal Flip
            if random.random() < 0.5:
                x = TF.hflip(x)
            # Random Rotation
            if random.random() < self.p:
                angle = random.uniform(-45, 45)
                x = TF.rotate(x, angle, interpolation=transforms.InterpolationMode.BILINEAR)
            # Random Scaling
            if random.random() < self.p:
                scale_factor = random.uniform(0.8, 1.2)
                h, w = x.shape[1:]
                new_h, new_w = int(h * scale_factor), int(w * scale_factor)
                x = TF.resize(x, (new_h, new_w))
                x = TF.center_crop(x, (h, w))  # Keep original size after scaling
            # Gaussian Blur (from OpenCV)
            if random.random() < self.p:
                x = TF.gaussian_blur(x, kernel_size=7, sigma=random.uniform(2, 7))
            # Random Translation
            if random.random() < self.p:
                max_dx = 20
                max_dy = 20
                dx = random.randint(-max_dx, max_dx)
                dy = random.randint(-max_dy, max_dy)
                x = TF.affine(x, angle=0, translate=[dx, dy], scale=1, shear=[0, 0])
            # Image corruption with thin lines
            if random.random() < self.p:
                num_lines = random.randint(2, 9)
                x = self.draw_random_lines(x, num_lines)
        return x
    
    def draw_random_lines(self, x, num_lines):
        # x: tensor of shape [1, H, W]
        _, h, w = x.shape
        center_x = random.randint(0, w-1)
        center_y = random.randint(0, h-1)

        img_np = (x.squeeze(0).cpu().numpy() * 255).astype(np.uint8)
        for _ in range(num_lines):
            angle = random.uniform(0, 360)
            length = random.randint(10, 50)
            x1 = int(center_x + length * np.cos(np.deg2rad(angle)))
            y1 = int(center_y + length * np.sin(np.deg2rad(angle)))
            x1 = np.clip(x1, 0, w-1)
            y1 = np.clip(y1, 0, h-1)
            cv2.line(img_np, (center_x, center_y), (x1, y1), (255,), thickness=1)
        img_np = img_np / 255.0
        return torch.from_numpy(img_np).unsqueeze(0).float()

**Subject-Based Training**

In [ ]:
def train_model_by_subject(root_dir, train_subjects, val_subjects, model=None, num_epochs=20, batch_size=16, lr=1e-3):
    # ---------------------
    # Augmentations for TRAINING
    # ---------------------
    train_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        RandomAugmentations(p=0.2),  # your requested augmentations
    ])

    # ---------------------
    # No augmentations for VALIDATION
    # ---------------------
    val_transform = transforms.Compose([
        transforms.Resize((64, 64)),
    ])

    train_dataset = EyeBoundingBoxDataset(train_subjects, root_dir, transform=train_transform, apply_preprocessing=True)
    val_dataset = EyeBoundingBoxDataset(val_subjects, root_dir, transform=val_transform, apply_preprocessing=True)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    device = next(model.parameters()).device if model else torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if model is None:
        model = LightweightBBoxCNN().to(device)
    else:
        model.to(device)

    torch.autograd.set_detect_anomaly(True)

    criterion = nn.SmoothL1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        avg_train_loss = train_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, targets).item() * inputs.size(0)

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(avg_val_loss)

        if (epoch + 1) % 5 == 0:
            torch.save(model.state_dict(), f'checkpoints/ppaug_cp_e{epoch + 1}')
            print('Checkpoint saved...')

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")

    return model, train_losses, val_losses

In [ ]:
# Parameters
BATCH_SIZE = 4
HIDDEN_SIZE = 64
LEARNING_RATE = 0.01
NUM_EPOCHS = 10
SUBJECT_SPLIT_FILE = 'subject_split.txt'  # you can change this if you want

# 1. Split subjects and save split
train_subjects, val_subjects, test_subjects = split_subjects_and_save(ROOT_DIR, output_file=SUBJECT_SPLIT_FILE)

# 2. Initialize model with optimized hidden layer size
model = LightweightBBoxCNN(hidden_size=HIDDEN_SIZE)

# 3. Train the model (with training-time augmentations only)
model, train_losses, val_losses = train_model_by_subject(
    root_dir=ROOT_DIR,
    train_subjects=train_subjects,
    val_subjects=val_subjects,
    model=model,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE
)